In [1]:
import torch
import numpy as np
import torchvision
import torchvision.transforms as transforms

## Fixed point module path
***
- Set the relative path to the following module

In [2]:
import sys
sys.path.append('../../../')
import ModelFxp12 as fxp

## CNN Model
***
- Put the model architecture here

In [3]:
class Net(torch.nn.Module):
    def __init__(self, num_classes = 3):
        torch.nn.Module.__init__(self)
        self.conv1 = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, padding=1),
            torch.nn.MaxPool2d(2,2),
            torch.nn.ReLU()
        )
        self.conv2 = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=4, out_channels=4, kernel_size=3, padding=1),
            torch.nn.MaxPool2d(2,2),
            torch.nn.ReLU()
        )
        self.conv3 = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=4, out_channels=4, kernel_size=3, padding=1),
            torch.nn.MaxPool2d(2,2),
            torch.nn.ReLU()
        )
        self.fc1 = torch.nn.Linear(in_features=4 * 3 * 3, out_features=10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        print(x.shape)
        x = x.view(x.size()[0], -1)
        x = self.fc1(x)
        return x

## Fixed point model
***
- Create here the fixed point version of the CNN model
- Parameters from pretrained models can be read as follows:
```python
cnn_model.load_state_dict(torch.load('parameters.pt', map_location=torch.device('cpu')))
```

In [4]:
weight_shift = 7
bias_shift = 7

cnn_model = Net()

cnn_model.load_state_dict(torch.load('train2_qat_sh7.pt', map_location=torch.device('cpu')))

layers = [
    # conv1
    fxp.Conv(cnn_model.conv1[0], weight_shift, bias_shift),
    fxp.MaxPool2d(2),
    fxp.ReLU(),

    # conv2
    fxp.Conv(cnn_model.conv2[0], weight_shift, bias_shift),
    fxp.MaxPool2d(2),
    fxp.ReLU(),

    # conv3
    fxp.Conv(cnn_model.conv3[0], weight_shift, bias_shift),
    fxp.MaxPool2d(2),
    fxp.ReLU(),
    
    # fc1
    fxp.Linear(cnn_model.fc1, weight_shift, bias_shift)       
]

cnn_model_fxp = fxp.ModelFxp(layers)

layers[0]: Conv
layers[1]: MaxPool2d
layers[2]: ReLU
layers[3]: Conv
layers[4]: MaxPool2d
layers[5]: ReLU
layers[6]: Conv
layers[7]: MaxPool2d
layers[8]: ReLU
layers[9]: Linear


## Input generation
***
- Generates an input according to the CNN input dimensions. 

In [5]:
batch = 1
channels = 1
height = 28
width = 28
input = torch.rand(batch, channels , height, width)

out = cnn_model_fxp(input) # 'input' scaled by the model, inside Forward() function

vhex = np.vectorize(hex)
print(f"Model output: {vhex(out.astype('uint32'))}")

Model output: [['0x298' '0x5a' '0x165' '0xffffff2e' '0x14d' '0x2a9' '0x6a' '0xffffffb1'
  '0x5c1' '0xe2']]


## Input memory image generation for VHDL simulation
***
- Scales the input and converts to integer
- Generates 'input_int.txt' file used as memory image for VHDL simulation

In [6]:
input_int = (input * (2 ** bias_shift)).numpy() # Scales the input and converts to integer
input_int = input_int.astype('uint8')

file = open("input_int.txt", "w+")

hex_chars = 3 # Number of hex characters in each file line. Set according to memory width 
              # memory width = hex_chars * 4

for p in input_int.flatten():    
    file.write('{0:0{1}X}\n'.format(p, hex_chars))

file.close()

## VHDL hardwired parameters generation
***
- Generates
    1. SkyNet_pkg.vhd file
    2. 'raw' directory with the memory images of parameter memories
- Function *GenerateHardwiredParametersVHDL()* requires an input according to the CNN input dimensions

In [7]:
cnn_model_fxp.GenerateHardwiredParametersVHDL(input_int, directory='HardwiredParameters_VHDL')

*** CONVOLUTIONAL LAYERS ***

sortedConvLayers (index, filters): [(0, 4), (1, 4), (2, 4)]
self.maxFilters: 4, convLayer[0]

FILTER_WEIGHTS[0]: begin of convolution layer 0 filters
FILTER_WEIGHTS[2]: convolution layer 0 filter bias
FILTER_WEIGHTS[3]: begin of convolution layer 1 filters
FILTER_WEIGHTS[8]: convolution layer 1 filter bias
FILTER_WEIGHTS[9]: begin of convolution layer 2 filters
FILTER_WEIGHTS[14]: convolution layer 2 filter bias

Data width: 32
Required filter memories depth: 15 (480 bits). BRAM depth: 16 (512 bits) (FILTERS_ADDR_WIDTH = 4)
Total required memory: 4 x 480 = 1920 bits. Total BRAM: 4 x 512 = 2048 bits



*** FULL CONNECTED LAYERS ***


Data width: 32
Required neuron memories depth: 60 (1920 bits). BRAM depth: 64 (2048 bits) (NEURONS_ADDR_WIDTH = 6)
Total required memory: 1 * 1920 = 1920 bits. Total BRAM: 1 * 2048 = 2048 bits

input shape: (1, 1, 28, 28)
<class 'ModelFxp12.Conv'>: out shape: (1, 4, 28, 28)
<class 'ModelFxp12.MaxPool2d'>: out shape: (1, 4, 14, 

## VHDL reconfigurable parameters generation
***
- Generates
    1. ReCoNNet_pkg.vhd file
    2. 'raw' directory with the memory images of parameter memories
    3. CNN_params.txt file with the reconfigurable CNN parameters
- Function *GenerateReCoNNetParametersVHDL()* requires an input according to the CNN input dimensions

In [8]:
cnn_model_fxp.GenerateReCoNNetParametersVHDL(input_int, directory='ReCoNNetParameters_VHDL')

*** CONVOLUTIONAL LAYERS ***

sortedConvLayers (index, filters): [(0, 4), (1, 4), (2, 4)]
self.maxFilters: 4, convLayer[0]

FILTER_WEIGHTS[0]: begin of convolution layer 0 filters
FILTER_WEIGHTS[2]: convolution layer 0 filter bias
FILTER_WEIGHTS[3]: begin of convolution layer 1 filters
FILTER_WEIGHTS[8]: convolution layer 1 filter bias
FILTER_WEIGHTS[9]: begin of convolution layer 2 filters
FILTER_WEIGHTS[14]: convolution layer 2 filter bias

Data width: 32
Required filter memories depth: 15 (480 bits). BRAM depth: 16 (512 bits) (FILTERS_ADDR_WIDTH = 4)
Total required memory: 4 x 480 = 1920 bits. Total BRAM: 4 x 512 = 2048 bits



*** FULL CONNECTED LAYERS ***


Data width: 32
Required neuron memories depth: 60 (1920 bits). BRAM depth: 64 (2048 bits) (NEURONS_ADDR_WIDTH = 6)
Total required memory: 1 * 1920 = 1920 bits. Total BRAM: 1 * 2048 = 2048 bits

input shape: (1, 1, 28, 28)
<class 'ModelFxp12.Conv'>: out shape: (1, 4, 28, 28)
<class 'ModelFxp12.MaxPool2d'>: out shape: (1, 4, 14, 

## C parameters generation
***
- Generates
    1. input.h file containing the input values scaled according to 'bias_shift'
    2. parameters.h file with the CNN parameters
- Function *GenerateParametersC()* requires an **scaled** input according to the CNN input dimensions

In [9]:
cnn_model_fxp.GenerateParametersC(input_int[0], directory='SoftwareParameters_C')

C parameters successfully generated!!!
	- parameters.h
	- input.h

Estimated data memory requirements:
	RAM
		- array parameters[7]: 392 Bytes
		- shared_fmap: 7200 Bytes
		- array acc[4] (LayerConvolution): 8 Bytes
		- array neurons[10] (FullConnected): 20 Bytes
		- input: 1568 Bytes (input_type: int16_t)
		- Total: 9188 bytes (8.97265625 KiB)
	ROM
		-total_conv_weights: 324 Bytes
		-total_conv_bias: 24 Bytes
		-total_fc_weights: 360 Bytes
		-total_fc_bias: 20 Bytes
		- Total: 728 bytes (0.7109375 KiB)
